# M-κ Verification — Thin-Wall T-Section with CFRP Reinforcement (CRC)

Verification notebook for `MKappaAnalysis` applied to a **carbon reinforced concrete (CRC)**
thin-wall T-section — the structural typology relevant for ultra-thin floor systems.

**Cross-section:**
- Web: b_w = 50 mm, h_w = 160 mm  
- Flange (top): h_f = 40 mm, b_f parametric (50 → 400 mm)  
- Total height: h = 200 mm

**Reinforcement:** CFRP bars — linear-elastic, brittle failure (no yielding plateau).

**Goals:**
1. Characterise the brittle CFRP stress-strain response vs ductile steel.
2. Confirm correct M-κ shape for a CRC thin-wall T-section.
3. Demonstrate flange width effect on capacity and post-peak behavior.
4. Verify solver robustness for brittle reinforcement (no yielding, sharp failure).

**Coordinate conventions:**
- `z` in `ReinforcementLayer` = distance from the *bottom* fibre  
- Positive moment = sagging; bottom fibre in tension  
- CFRP carries tension only (no compression capacity)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Core scite imports
from scite.cs_design.shapes import RectangularShape, TShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import SteelReinforcement, create_steel
from scite.matmod.carbon_reinforcement import CarbonReinforcement, create_carbon
from scite.mkappa.mkappa import MKappaAnalysis

print("Imports OK")

In [ ]:
def print_convergence_report(mk: MKappaAnalysis, label: str = ""):
    """Print a compact convergence and failure report for a solved MKappaAnalysis."""
    if mk.M_values is None or len(mk.M_values) == 0:
        print(f"  {label}: No data.")
        return
    n_pts = len(mk.M_values)
    N_abs = np.abs(mk.N_values)
    idx_peak = int(np.argmax(mk.M_values))
    tag = f"  [{label}]" if label else " "
    print(f"{tag}")
    print(f"    Points solved  : {n_pts} / {mk.n_kappa}")
    print(f"    M_peak         : {mk.M_values[idx_peak]:.2f} kN·m")
    print(f"    κ at peak      : {mk.kappa_values[idx_peak]*1000:.4f} ‰/mm")
    print(f"    ε_bot at peak  : {mk.eps_bot_values[idx_peak]:.5f}")
    print(f"    |N| max resid  : {N_abs.max():.4f} kN  (tol={mk.N_tol:.4f} kN)")
    print(f"    |N| mean resid : {N_abs.mean():.4f} kN")

print("Helper defined.")

---
## 1. Material Characterisation — CFRP vs Steel

CFRP bars exhibit **linear-elastic, brittle** behaviour: the stress rises linearly
until fibre rupture with no yielding plateau. A numerical post-peak softening tail
(slope = `post_peak_factor × E`) is introduced solely for solver convergence.

Compare three typical CFRP grades (`C1500`, `C2000`, `C2500`) against conventional
B500B steel to highlight the key differences:
- Much higher elastic modulus ratio E_f / E_s ≈ 1.0–1.15 (similar stiffness)  
- 3–5× higher strength, but brittle failure  
- No yielding plateau → M-κ will **not** show a horizontal yield branch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_sig, ax_tab = axes

steel = create_steel('B500B')
carbon_grades = ['C1500', 'C2000', 'C2500']
colors_cfrp = ['#c0392b', '#e67e22', '#8e44ad']

# --- stress-strain curves ---
# Steel (piecewise: elastic + yield plateau + hardening, simplified bilinear)
eps_s = np.linspace(-0.015, 0.06, 500)
sig_s = steel.get_sig(eps_s)
ax_sig.plot(eps_s * 1000, sig_s, 'b-', linewidth=2.0, label=f'B500B  (f_yd={steel.f_yd:.0f} MPa)')

for grade, color in zip(carbon_grades, colors_cfrp):
    c = create_carbon(grade)
    eps_range = c.get_eps_plot_range()
    sig_range = c.get_sig(eps_range)
    ax_sig.plot(eps_range * 1000, sig_range, color=color, linewidth=2.0,
                label=f'{grade}  f_t={c.f_t:.0f} MPa, E={c.E/1000:.0f} GPa')
    ax_sig.axvline(c.eps_cr * 1000, color=color, linestyle=':', alpha=0.5, linewidth=1.0)

ax_sig.set_xlabel('Strain ε [‰]', fontsize=11)
ax_sig.set_ylabel('Stress σ [MPa]', fontsize=11)
ax_sig.set_title('Stress-Strain: CFRP grades vs B500B Steel', fontsize=11)
ax_sig.legend(fontsize=9)
ax_sig.grid(True, alpha=0.3)
ax_sig.set_xlim(-2, 20)

# --- summary table ---
ax_tab.axis('off')
headers = ['Material', 'E [GPa]', 'f_t / f_y [MPa]', 'ε_fail [‰]', 'Behaviour']
rows = []
rows.append(['B500B steel', f'{steel.E_s/1000:.0f}', f'{steel.f_yd:.0f}',
             f'{steel.eps_yd*1000:.2f} (yield)', 'Ductile'])
for grade in carbon_grades:
    c = create_carbon(grade)
    rows.append([grade, f'{c.E/1000:.0f}', f'{c.f_t:.0f}',
                 f'{c.eps_cr*1000:.2f}', 'Brittle'])
table = ax_tab.table(cellText=rows, colLabels=headers,
                     cellLoc='center', loc='center', bbox=[0, 0.2, 1, 0.7])
table.auto_set_font_size(False)
table.set_fontsize(9)
ax_tab.set_title('Material Properties Summary', fontsize=11, pad=20)

plt.suptitle('CFRP vs Steel — Key Differences for M-κ Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Base CRC T-Section — Single Analysis

**Reference geometry** (used throughout this notebook unless stated otherwise):

| Parameter | Value |
|-----------|-------|
| Web width b_w | 50 mm |
| Web height h_w | 160 mm |
| Flange height h_f | 40 mm |
| Flange width b_f | 200 mm (reference) |
| Total height h | 200 mm |
| Concrete | C30/37, α_cc=0.85, γ_c=1.5 |
| Reinforcement | CFRP C2000, A_s = 100 mm², z = 20 mm |

A single CFRP layer at z = 20 mm from the bottom. The thin cover is feasible with
non-corroding CFRP bars.

In [ ]:
# ── Reference geometry ──────────────────────────────────────────────────────
b_w_ref, h_w_ref = 50.0, 160.0
h_f_ref, b_f_ref = 40.0, 200.0
h_total_ref = h_w_ref + h_f_ref  # = 200 mm
z_reinf = 20.0   # mm from bottom (thin CFRP cover)
A_s_ref  = 100.5 # mm²  (≈ 2Ø8 CFRP bars)

shape_T_crc = TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f_ref, h_f=h_f_ref)
cfrp_C2000  = create_carbon('C2000')
concrete_C30 = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)

print(f"Concrete C30/37: f_cd={concrete_C30.f_cd:.2f} MPa, "
      f"ε_c2={concrete_C30.eps_c2_computed:.4f}, ε_cu2={concrete_C30.eps_cu2_computed:.4f}")
print(f"CFRP C2000 : E={cfrp_C2000.E:.0f} MPa, f_t={cfrp_C2000.f_t:.0f} MPa, "
      f"ε_cr={cfrp_C2000.eps_cr*1000:.3f} ‰")
print(f"")
print(f"T-section geometry:")
print(f"  h_total = {shape_T_crc.h_total:.0f} mm")
print(f"  A_c     = {shape_T_crc.area:.0f} mm²")
print(f"  centroid from bottom = {shape_T_crc.centroid_y:.1f} mm")

layer_cfrp = ReinforcementLayer(
    z=z_reinf, A_s=A_s_ref, material=cfrp_C2000, name="CFRP C2000 bottom"
)
rein_crc = ReinforcementLayout(layers=[layer_cfrp])

cs_T_crc = CrossSection(
    shape=shape_T_crc,
    concrete=concrete_C30,
    reinforcement=rein_crc,
    n_points=200,
)

print(f"")
print(f"Cross-section properties:")
print(f"  Shape    : {type(cs_T_crc.shape).__name__}")
print(f"  h_total  : {cs_T_crc.h_total:.0f} mm")
print(f"  A_c      : {cs_T_crc.A_c:,.0f} mm²")
print(f"  A_s      : {cs_T_crc.A_s:.1f} mm²")
print(f"  ρ (gross): {cs_T_crc.reinforcement_ratio*100:.3f} %")
print(f"  ρ (web)  : {A_s_ref / (b_w_ref * (h_total_ref - z_reinf)) * 100:.3f} %")

In [ ]:
# ── Solve M-κ (reference CRC T-section) ────────────────────────────────────
mk_T_crc = MKappaAnalysis(cs=cs_T_crc, n_kappa=150, N_Ed=0.0)
mk_T_crc.solve()

print("CRC T-section — convergence report:")
print_convergence_report(mk_T_crc, label="b_f=200 mm, CFRP C2000, A_s=100 mm²")

In [ ]:
mk_T_crc.plot_summary(
    "CRC T-Section — b_w=50, h_w=160, b_f=200, h_f=40 mm | C30/37, CFRP C2000"
)

---
## 3. Effect of Flange Width b_f — with Failure Mode Identification

Sweep `b_f` from 50 mm (= b_w, degenerate rectangle) up to 400 mm.  
All other parameters fixed (b_w=50, h_w=160, h_f=40, C30/37, CFRP C2000, A_s=100 mm²).

**Markers at peak** indicate which material reached its strain limit first:

| Marker | Mode | Criterion |
|--------|------|-----------|
| ◆ | Concrete crushing | η_c = \|ε_top\| / ε_cu2 > η_f |
| × | CFRP rupture | η_f = ε_bot / ε_cr > η_c |
| ○ | Balanced | η_c ≈ η_f (within 5 %) |

**Physical interpretation of the transition:**

- **Small b_f** → narrow compression zone; to balance the CFRP tension force
  (~200 kN at ε_cr = 12 ‰) the concrete would need a depth exceeding h —
  geometrically impossible. So the top fibre crushes (ε_cu2 = 3.5 ‰) at a
  curvature where the CFRP is still well below its rupture strain.
  **Concrete governs.**

- **Large b_f** → the wide flange supplies large compression area at low stress;
  equilibrium is satisfied with a shallow neutral axis, leaving room for the
  bottom CFRP to accumulate large tensile strain. The CFRP reaches ε_cr before
  the concrete top fibre reaches ε_cu2.
  **CFRP governs.**

The mode transition (balanced condition) occurs near b_f ≈ 200–300 mm for
these material and geometry parameters — visible in the table as η_c ≈ η_f.


In [ ]:
b_f_values = [50.0, 100.0, 150.0, 200.0, 300.0, 400.0]  # mm

fig, ax = plt.subplots(figsize=(11, 6.5))
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(b_f_values)))

bf_results = []

for b_f, color in zip(b_f_values, colors):
    shape = TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f, h_f=h_f_ref)
    cs = CrossSection(
        shape=shape,
        concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=z_reinf, A_s=A_s_ref, material=create_carbon('C2000'))]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=150, N_Ed=0.0)
    mk.solve()

    M_peak = mk.M_values.max() if mk.M_values is not None and len(mk.M_values) > 0 else float('nan')
    label_str = f"b_f={b_f:.0f} mm" + (" (rect)" if b_f == b_w_ref else "")
    mk.plot_mk(ax=ax, color=color, linewidth=2.0,
               label=f"{label_str}  M_pk={M_peak:.1f} kN·m",
               show_failure_marker=True)

    fm = mk.get_failure_mode()
    bf_results.append((b_f, len(mk.M_values) if mk.M_values is not None else 0, M_peak, fm))

MKappaAnalysis.add_failure_mode_legend(ax, fontsize=8, ncol=2)
ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title(
    "M-κ — Effect of Flange Width (CRC T-section)\n"
    "b_w=50, h_w=160, h_f=40 mm | C30/37, CFRP C2000, A_s=100 mm²\n"
    "Markers at peak: ◆ concrete crushing   × CFRP rupture   ○ balanced",
    fontsize=10
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Tabular summary with utilisation ratios
print(f"\n{'b_f [mm]':>10} {'M_peak':>10} {'mode':>15} {'η_c':>7} {'η_f':>7}")
print("-" * 55)
for b_f, n_pts, M_peak, fm in bf_results:
    tag = "  (rect)" if b_f == b_w_ref else ""
    print(f"{b_f:>10.0f} {M_peak:>10.2f} {fm['mode']:>15} "
          f"{fm['eta_c']:>7.3f} {fm['eta_f']:>7.3f}{tag}")

---
## 4. CFRP vs Steel — Direct Comparison (Same Section)

Compare the **CRC** version (CFRP C2000) with a **conventional RC** version (B500B steel)
for the same T-section geometry and reinforcement area.  

Key expected observations:
- RC: long yielding plateau (ductile), gradual M drop after concrete crushing  
- CRC: steeper linear rise, higher M_peak (higher strength), abrupt failure (brittle)

In [ ]:
cs_T_rc = CrossSection(
    shape=TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f_ref, h_f=h_f_ref),
    concrete=concrete_C30,
    reinforcement=ReinforcementLayout(
        layers=[ReinforcementLayer(z=z_reinf, A_s=A_s_ref, material=create_steel('B500B'))]
    ),
    n_points=200,
)
mk_T_rc = MKappaAnalysis(cs=cs_T_rc, n_kappa=150, N_Ed=0.0)
mk_T_rc.solve()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax_abs, ax_norm = axes

cases = [
    (mk_T_crc, "CRC — CFRP C2000",  '#c0392b', '-'),
    (mk_T_rc,  "RC  — B500B steel",  '#2980b9', '--'),
]

for mk, label, color, ls in cases:
    if mk.M_values is None or len(mk.M_values) == 0:
        continue
    kappa_pm = mk.kappa_values * 1000.0
    M_peak = mk.M_values.max()
    kappa_peak = kappa_pm[mk.M_values.argmax()]

    # Absolute M-κ: use mk.plot_mk so show_failure_marker is supported;
    # linestyle is passed via plot_kwargs workaround until plot_mk gains ls param.
    ax_abs.plot(kappa_pm, mk.M_values, color=color, linestyle=ls, linewidth=2.2,
                label=f"{label}  M_pk={M_peak:.1f} kN·m")
    mk._add_failure_marker(ax_abs, color=color)

    ax_norm.plot(kappa_pm / kappa_peak, mk.M_values / M_peak,
                 color=color, linestyle=ls, linewidth=2.2,
                 label=f"{label}\n(M_pk={M_peak:.1f} kN·m, κ_pk={kappa_peak:.3f} 1/m)")

MKappaAnalysis.add_failure_mode_legend(ax_abs, fontsize=9)

for ax, xlabel, ylabel, title in [
    (ax_abs,  "κ [1/m]", "M [kN·m]",       "Absolute M-κ"),
    (ax_norm, "κ / κ_peak [-]", "M / M_peak [-]", "Normalised — ductility comparison"),
]:
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "CRC (CFRP) vs RC (Steel) — b_w=50, h_w=160, b_f=200, h_f=40 mm, C30/37, A_s=100 mm²",
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("\nConvergence reports:")
print_convergence_report(mk_T_crc, label="CRC — CFRP C2000")
print_convergence_report(mk_T_rc,  label="RC  — B500B steel")

---
## 5. Effect of CFRP Reinforcement Area A_s

With steel, more reinforcement shifts the failure from concrete-governed to
steel-governed, producing a characteristic change in ductility.

With **brittle CFRP**, increasing A_s raises M_peak but there is still no yield
plateau — the failure remains sudden regardless of reinforcement ratio.
The balanced reinforcement ratio concept differs from RC.

In [ ]:
# A_s sweep: from lightly to heavily reinforced web
# For b_w=50, h_w=160: web area = 50*160 = 8000 mm²
A_s_values = [25.0, 50.0, 100.0, 200.0, 400.0, 700.0]  # mm²

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(A_s_values)))
As_results = []

for A_s, color in zip(A_s_values, colors):
    rho_web = A_s / (b_w_ref * (h_total_ref - z_reinf)) * 100  # % w.r.t. web
    cs = CrossSection(
        shape=TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f_ref, h_f=h_f_ref),
        concrete=EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=z_reinf, A_s=A_s, material=create_carbon('C2000'))]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=150, N_Ed=0.0)
    mk.solve()

    M_peak = mk.M_values.max() if mk.M_values is not None and len(mk.M_values) > 0 else float('nan')
    mk.plot_mk(ax=ax, color=color, linewidth=2.0,
               label=f"A_s={A_s:.0f} mm² (ρ={rho_web:.2f}%)  M_pk={M_peak:.1f} kN·m",
               show_failure_marker=True)

    fm = mk.get_failure_mode()
    As_results.append((A_s, rho_web, len(mk.M_values) if mk.M_values is not None else 0, M_peak, fm))

MKappaAnalysis.add_failure_mode_legend(ax, fontsize=8, ncol=2, loc='upper left')
ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title(
    "M-κ — Effect of CFRP Reinforcement Area (CRC T-section)\n"
    "b_w=50, h_w=160, b_f=200, h_f=40 mm | C30/37, CFRP C2000\n"
    "Markers at peak: ◆ concrete crushing   × CFRP rupture   ○ balanced",
    fontsize=10
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'A_s [mm²]':>10} {'ρ_web [%]':>10} {'Pts':>5} {'M_peak [kN·m]':>14} "
      f"{'mode':>15} {'η_c':>7} {'η_f':>7}")
print("-" * 75)
for A_s, rho_web, n_pts, M_peak, fm in As_results:
    print(f"{A_s:>10.0f} {rho_web:>10.3f} {n_pts:>5} {M_peak:>14.2f} "
          f"{fm['mode']:>15} {fm['eta_c']:>7.3f} {fm['eta_f']:>7.3f}")

---
## 6. Solver Robustness — Brittle CFRP Reinforcement

Brittle reinforcement creates the most challenging convergence conditions:
- No yielding plateau means the bracket for `brentq` must straddle a sharp force drop  
- Very thin webs produce small concrete compression forces near failure  
- Mixed grades (C1500 vs C2500) affect the ε_cr constraint on solver bounds

Test matrix:
- **Varying reinforcement area** (under → over-reinforced for CFRP)
- **Varying concrete grade** (C20 → C60) with fixed CFRP
- **Varying CFRP grade** (C1500, C2000, C2500)
- **Flange width extremes** (b_f = b_w to b_f = 400 mm)

In [ ]:
print("=== Robustness Checks — CFRP T-section ===")
print(f"Base geometry: b_w={b_w_ref:.0f}, h_w={h_w_ref:.0f}, b_f={b_f_ref:.0f}, h_f={h_f_ref:.0f} mm\n")

robustness_cases = [
    # label                               b_f     A_s     f_ck   grade
    ("Very low ρ    A_s=10 mm²",          200.0,  10.0,   30.0,  'C2000'),
    ("Low ρ         A_s=50 mm²",          200.0,  50.0,   30.0,  'C2000'),
    ("Medium ρ      A_s=100 mm²",         200.0, 100.0,   30.0,  'C2000'),
    ("High ρ        A_s=300 mm²",         200.0, 300.0,   30.0,  'C2000'),
    ("Very high ρ   A_s=700 mm²",         200.0, 700.0,   30.0,  'C2000'),
    ("C20 concrete  A_s=100 mm²",         200.0, 100.0,   20.0,  'C2000'),
    ("C45 concrete  A_s=100 mm²",         200.0, 100.0,   45.0,  'C2000'),
    ("C60 concrete  A_s=100 mm²",         200.0, 100.0,   60.0,  'C2000'),
    ("Grade C1500   A_s=100 mm²",         200.0, 100.0,   30.0,  'C1500'),
    ("Grade C2500   A_s=100 mm²",         200.0, 100.0,   30.0,  'C2500'),
    ("Rect (b_f=b_w=50)",                  50.0, 100.0,   30.0,  'C2000'),
    ("Wide flange   b_f=400 mm",          400.0, 100.0,   30.0,  'C2000'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax_As   = axes[0, 0]   # A_s variation
ax_fck  = axes[0, 1]   # concrete grade
ax_gr   = axes[1, 0]   # CFRP grade
ax_bf   = axes[1, 1]   # flange width

groups = {
    'A_s variation':     ([0,1,2,3,4], ax_As,  plt.cm.viridis),
    'Concrete grade':    ([5,6,7],     ax_fck, plt.cm.Reds),
    'CFRP grade':        ([8,9,2],     ax_gr,  plt.cm.cool),
    'Flange width':      ([10,11,2],   ax_bf,  plt.cm.plasma),
}

results_table = []

for label, b_f, A_s, f_ck, grade in robustness_cases:
    cs = CrossSection(
        shape=TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f, h_f=h_f_ref),
        concrete=EC2ParabolaRectangle(f_ck=f_ck, alpha_cc=0.85, gamma_c=1.5),
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=z_reinf, A_s=A_s, material=create_carbon(grade))]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=150, N_Ed=0.0)
    try:
        mk.solve()
        n_pts  = len(mk.M_values) if mk.M_values is not None else 0
        M_peak = mk.M_values.max() if n_pts > 0 else float('nan')
        N_max  = np.abs(mk.N_values).max() if n_pts > 0 else float('nan')
        status = "OK" if n_pts >= 5 else "WARN"
        results_table.append((label, b_f, A_s, f_ck, grade, n_pts, M_peak, N_max, status, mk))
    except Exception as e:
        results_table.append((label, b_f, A_s, f_ck, grade, 0, float('nan'), float('nan'),
                               f"ERROR: {e}", None))

# --- draw into subplot groups ---
for group_name, (indices, ax, cmap) in groups.items():
    colors_g = cmap(np.linspace(0.2, 0.9, len(indices)))
    for color, idx in zip(colors_g, indices):
        row = results_table[idx]
        mk_r = row[-1]
        if mk_r is not None and mk_r.M_values is not None and len(mk_r.M_values) > 0:
            short = row[0].split()[0] + ' ' + row[0].split()[-1]
            ax.plot(mk_r.kappa_values * 1000.0, mk_r.M_values,
                    color=color, linewidth=1.8,
                    label=f"{row[0]}  M_pk={row[6]:.1f}")
    ax.set_xlabel("κ [1/m]", fontsize=9)
    ax.set_ylabel("M [kN·m]", fontsize=9)
    ax.set_title(group_name, fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle("Robustness — CRC T-section (b_w=50, h_w=160, h_f=40 mm)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# --- text table ---
print(f"\n{'Case':<36} {'b_f':>6} {'A_s':>6} {'f_ck':>5} {'Grade':>6} "
      f"{'Pts':>4} {'M_peak':>9} {'|N|_max':>9} {'Status':>8}")
print("-" * 98)
for row in results_table:
    label, b_f, A_s, f_ck, grade, n_pts, M_peak, N_max, status, _ = row
    print(f"{label:<36} {b_f:>6.0f} {A_s:>6.0f} {f_ck:>5.0f} {grade:>6} "
          f"{n_pts:>4} {M_peak:>9.2f} {N_max:>9.4f} {status:>8}")

---
## 7. CFRP Grade Comparison — C1500 / C2000 / C2500

Show the full M-κ curves for all three CFRP grades side by side and compare with
the steel reference. At equal A_s, the higher grade gives a higher M_peak but
an earlier fibre rupture at smaller κ.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

grade_cases = [
    ('C1500', '#e74c3c', '-'),
    ('C2000', '#8e44ad', '-'),
    ('C2500', '#2c3e50', '-'),
]
steel_case = ('B500B steel', '#2980b9', '--')

for grade, color, ls in grade_cases:
    c = create_carbon(grade)
    cs = CrossSection(
        shape=TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f_ref, h_f=h_f_ref),
        concrete=concrete_C30,
        reinforcement=ReinforcementLayout(
            layers=[ReinforcementLayer(z=z_reinf, A_s=A_s_ref, material=c)]
        ),
        n_points=200,
    )
    mk = MKappaAnalysis(cs=cs, n_kappa=150, N_Ed=0.0)
    mk.solve()
    if mk.M_values is not None and len(mk.M_values) > 0:
        M_peak = mk.M_values.max()
        ax.plot(mk.kappa_values * 1000.0, mk.M_values,
                color=color, linestyle=ls, linewidth=2.0,
                label=f"CFRP {grade}  (f_t={c.f_t:.0f} MPa)  M_pk={M_peak:.1f} kN·m")

# Add steel reference
cs_steel = CrossSection(
    shape=TShape(b_w=b_w_ref, h_w=h_w_ref, b_f=b_f_ref, h_f=h_f_ref),
    concrete=concrete_C30,
    reinforcement=ReinforcementLayout(
        layers=[ReinforcementLayer(z=z_reinf, A_s=A_s_ref, material=create_steel('B500B'))]
    ),
    n_points=200,
)
mk_steel = MKappaAnalysis(cs=cs_steel, n_kappa=150, N_Ed=0.0)
mk_steel.solve()
if mk_steel.M_values is not None and len(mk_steel.M_values) > 0:
    M_peak = mk_steel.M_values.max()
    ax.plot(mk_steel.kappa_values * 1000.0, mk_steel.M_values,
            color='#2980b9', linestyle='--', linewidth=2.0,
            label=f"B500B steel                    M_pk={M_peak:.1f} kN·m")

ax.set_xlabel("κ [1/m]", fontsize=12)
ax.set_ylabel("M [kN·m]", fontsize=12)
ax.set_title(
    "M-κ — CFRP Grade Comparison vs Steel (same section & area)\n"
    "b_w=50, h_w=160, b_f=200, h_f=40 mm | C30/37, A_s=100 mm²",
    fontsize=11
)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()